# 策略执行引擎教程

本教程介绍 open-xquant 的核心管道：**Universe → Indicator → Signal → Rule** 四阶段模型。

我们将用一个 SMA 均线交叉策略（SMA10 金叉/死叉 SMA50）作为贯穿全文的示例，逐层拆解每个阶段的工作方式：

- **Indicator** — 路径无关的纯函数计算（向量化）
- **Signal** — 跨 symbol 截面操作，生成方向性预测（向量化）
- **Rule** — 路径相关的状态机，读取持仓生成订单（逐 bar）
- **Engine** — 串联四阶段，通过三个 Protocol 接口驱动执行

### 三接口架构：一个引擎，三种模式

Engine 不知道自己在运行回测、模拟盘还是实盘——它只依赖三个 Protocol 接口：

| Protocol | 回测 | 模拟盘（未来） | 实盘（未来） |
|----------|------|---------------|-------------|
| `MarketDataProvider` | `LocalMarketDataProvider` | `RealtimeDataProvider` | `RealtimeDataProvider` |
| `Broker` | `SimBroker` | `SimBroker` | `BrokerAdapter` |

同一套策略代码，不改一行，只需更换 Provider 即可在回测、模拟盘、实盘之间切换。

## 1. 安装依赖

执行引擎是 open-xquant 核心功能，无需额外依赖。如需下载真实行情数据：

```bash
pip install open-xquant[yfinance]
```

---
## 2. Indicator — 技术指标

Indicator 是路径无关的纯函数：输入一个 symbol 的 DataFrame，输出等长 Series，引擎负责追加为宽表的新列。

SMA（简单移动平均线）是最基础的趋势指标：

In [1]:
import pandas as pd
from oxq.indicators import SMA

# 构造一段模拟行情
dates = pd.bdate_range("2024-01-01", periods=10)
mktdata = pd.DataFrame({
    "close": [100, 102, 101, 105, 108, 107, 110, 112, 109, 115],
}, index=dates)

sma = SMA()
result = sma.compute(mktdata, period=3)

mktdata["sma_3"] = result
print("SMA(3) 计算结果：")
print(mktdata[["close", "sma_3"]])

SMA(3) 计算结果：
            close       sma_3
2024-01-01    100         NaN
2024-01-02    102         NaN
2024-01-03    101  101.000000
2024-01-04    105  102.666667
2024-01-05    108  104.666667
2024-01-08    107  106.666667
2024-01-09    110  108.333333
2024-01-10    112  109.666667
2024-01-11    109  110.333333
2024-01-12    115  112.000000


前 `period - 1` 行是 NaN（滚动窗口不足），这是正常行为。Signal 和 Rule 层会处理这些 NaN。

**关键特性**：
- `compute` 是纯函数 — 不修改输入 DataFrame，不依赖外部状态
- 同一个 SMA 类可以用不同参数注册为多个实例（如 `sma_10` 和 `sma_50`）
- 默认对 `close` 列计算，也可以通过 `column` 参数指定其他列

验证 SMA 满足 Indicator Protocol：

In [2]:
from oxq.core import Indicator

print(f"SMA 满足 Indicator Protocol: {isinstance(SMA(), Indicator)}")
print(f"Indicator name: {SMA().name}")

SMA 满足 Indicator Protocol: True
Indicator name: SMA


---
## 3. Signal — 信号生成

Signal 描述「交易的欲望」——方向性预测，而非交易指令，表现为 True/False 序列，而不再是数值序列。与 Indicator 的关键区别：

| 维度 | Indicator | Signal |
|------|-----------|--------|
| 输入 | 单个 symbol 的 DataFrame | **全 universe** 的 mktdata |
| 输出 | 一个 Series | **每个 symbol** 各一个 Series |
| 视角 | per symbol | cross-sectional（截面） |

Crossover 信号检测快线上穿慢线的时刻：

In [3]:
from oxq.signals import Crossover

# 构造一组含有金叉的数据
dates = pd.bdate_range("2024-01-01", periods=6)
df = pd.DataFrame({
    "close": [100, 98, 97, 99, 102, 105],
    "sma_10": [99, 98, 97, 99, 101, 103],    # 快线
    "sma_50": [100, 100, 100, 100, 100, 100],  # 慢线
}, index=dates)

# Signal 接收整个 mktdata（dict），返回每个 symbol 的信号
mktdata_dict = {"AAPL": df}
crossover = Crossover()
signals = crossover.compute(mktdata_dict, fast="sma_10", slow="sma_50")

df["sma_10_x_sma_50"] = signals["AAPL"]
print("Crossover 信号：")
print(df[["sma_10", "sma_50", "sma_10_x_sma_50"]])

Crossover 信号：
            sma_10  sma_50  sma_10_x_sma_50
2024-01-01      99     100            False
2024-01-02      98     100            False
2024-01-03      97     100            False
2024-01-04      99     100            False
2024-01-05     101     100             True
2024-01-08     103     100            False


Day 4（2024-01-04）触发了上穿信号：前一天 sma_10(99) <= sma_50(100)，当天 sma_10(101) > sma_50(100)。

**Signal 为什么要接收全 universe？** 因为有些信号需要跨 symbol 操作——比如「按动量排名取 top 5」。Crossover 恰好只看单个 symbol，但 Protocol 为截面操作预留了能力。

---
## 4. Rule — 交易规则

Rule 是路径相关的——它知道当前持仓和资金状态，逐 bar 执行。

与 Indicator/Signal 的关键区别：

| 维度 | Indicator / Signal | Rule |
|------|-------------------|------|
| 输入 | 整个时间序列 | **单行**（当前 bar） |
| 状态 | 无状态（纯函数） | **有状态**（读取 Portfolio） |
| 输出 | Series | **Order \| None** |
| 计算模式 | 向量化 | 逐 bar 循环 |

In [4]:
from oxq.core import Portfolio, Position
from oxq.rules import EntryRule, ExitRule

# EntryRule: 信号触发 + 无持仓 → 买入
entry = EntryRule(signal="sma_10_x_sma_50", shares=100)

# 模拟一个 bar 的数据
row = pd.Series({"close": 102.0, "sma_10": 101.0, "sma_50": 100.0, "sma_10_x_sma_50": True})
portfolio = Portfolio(cash=100_000.0)

order = entry.evaluate("AAPL", row, portfolio)
print(f"EntryRule 产生订单: {order}")

EntryRule 产生订单: Order(symbol='AAPL', side='BUY', shares=100, order_type='market')


In [5]:
# ExitRule: 快线 < 慢线 + 有持仓 → 卖出
exit_rule = ExitRule(fast="sma_10", slow="sma_50")

row_exit = pd.Series({"close": 95.0, "sma_10": 97.0, "sma_50": 100.0})
portfolio_with_pos = Portfolio(
    cash=50_000.0,
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=102.0)},
)

sell_order = exit_rule.evaluate("AAPL", row_exit, portfolio_with_pos)
print(f"ExitRule 产生订单: {sell_order}")

ExitRule 产生订单: Order(symbol='AAPL', side='SELL', shares=100, order_type='market')


**执行优先级**：引擎每个 bar 先执行 ExitRule（平仓），再执行 EntryRule（开仓）。避免同一个 bar 既买又卖。

---
## 5. Strategy — 声明式策略定义

Strategy 将 Universe、Indicator、Signal、Rule 组合为一个完整的声明式管道：

```
Universe     → 确定标的池
  ↓
Indicator    → 计算指标列（sma_10, sma_50）
  ↓
Signal       → 生成信号列（sma_10_x_sma_50）
  ↓
Rule         → 读取信号 + 持仓 → 生成订单
```

In [6]:
from oxq.core import Strategy
from oxq.universe import StaticUniverse
from oxq.indicators import SMA
from oxq.signals import Crossover
from oxq.rules import EntryRule, ExitRule

strategy = Strategy(
    name="sma_crossover",
    hypothesis="短期均线上穿长期均线的标的在后续持有期内有正超额收益",
    universe=StaticUniverse(("AAPL",)),
    indicators={
        "sma_10": (SMA(), {"period": 10}),
        "sma_50": (SMA(), {"period": 50}),
    },
    signals={
        "sma_10_x_sma_50": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
    },
    entry_rules=[EntryRule(signal="sma_10_x_sma_50", shares=100)],
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

print(f"策略名称: {strategy.name}")
print(f"假设: {strategy.hypothesis}")
print(f"指标: {list(strategy.indicators.keys())}")
print(f"信号: {list(strategy.signals.keys())}")
print(f"入场规则: {len(strategy.entry_rules)} 条")
print(f"出场规则: {len(strategy.exit_rules)} 条")

策略名称: sma_crossover
假设: 短期均线上穿长期均线的标的在后续持有期内有正超额收益
指标: ['sma_10', 'sma_50']
信号: ['sma_10_x_sma_50']
入场规则: 1 条
出场规则: 1 条


策略定义是纯声明式的——它描述「做什么」，不关心「怎么做」。同一个 Strategy 对象可以在回测、模拟盘、实盘中执行，代码零修改。

---
## 6. 宽表数据模型

在运行引擎之前，先理解它如何处理数据。`mktdata` 是按 symbol 索引的 DataFrame 集合，各阶段通过**追加列**逐步加宽每个 symbol 的宽表：

```
原始行情              Indicator 后            Signal 后
+-----------+       +---------------+      +------------------+
| open      |       | open          |      | open             |
| high      |       | high          |      | high             |
| low       | ──▶  | low           | ──▶ | low              |
| close     |       | close         |      | close            |
| volume    |       | volume        |      | volume           |
|           |       | sma_10  (新增)|      | sma_10           |
|           |       | sma_50  (新增)|      | sma_50           |
|           |       |               |      | sma_10_x_sma_50  |
+-----------+       +---------------+      +------------------+
```

这种设计的好处：Signal 无需知道 Indicator 的输出格式，只需按列名引用；Rule 同理。所有中间结果在同一张表上可见可查。

---
## 7. 运行引擎

Engine 将四阶段串联执行。它不知道自己在运行回测——它只是通过 Provider 接口获取数据、提交订单、接收成交。

当我们传入 `LocalMarketDataProvider`（历史数据）+ `SimBroker`（模拟撮合），这就等于回测。

In [7]:
from oxq.data import YFinanceDownloader

# 下载 AAPL 2023-2024 两年数据
downloader = YFinanceDownloader()
path = downloader.download("AAPL", start="2023-01-01", end="2024-12-31")
print(f"数据已保存到: {path}")

数据已保存到: /Users/daodao/.oxq/data/market/AAPL.parquet


In [ ]:
from oxq.core import Engine
from oxq.data import LocalMarketDataProvider
from oxq.trade import SimBroker

# 选择 Provider：历史数据 + 模拟撮合 = 回测模式
market = LocalMarketDataProvider()
sim_broker = SimBroker()

engine = Engine()
result = engine.run(
    strategy,
    market=market,
    broker=sim_broker,
    start="2023-01-01",
    end="2024-12-31",
)

print(f"总收益率:   {result.total_return():.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio():.2f}")
print(f"最大回撤:   {result.max_drawdown():.2%}")
print(f"交易次数:   {len(result.trades)}")

引擎内部执行流程：

1. **Phase 0 — Universe**：从 `StaticUniverse` 获取标的列表 `["AAPL"]`
2. **Phase 1 — Indicator**：对 AAPL 的 DataFrame 调用 `SMA.compute()`，追加 `sma_10`、`sma_50` 两列
3. **Phase 2 — Signal**：调用 `Crossover.compute(mktdata)`，追加 `sma_10_x_sma_50` 布尔列
4. **Phase 3 — Rule**：逐 bar 遍历，对每个 bar 先执行 ExitRule 再执行 EntryRule，通过 `broker` 提交订单并接收成交

---
## 8. 查看交易记录

`result.trades` 包含所有成交记录（`Fill` 对象）：

In [9]:
if result.trades:
    print(f"{'日期':<25} {'方向':>4}  {'数量':>4}  {'标的':<6} {'成交价':>8}")
    print("-" * 55)
    for fill in result.trades:
        print(
            f"{fill.filled_at:<25} {fill.order.side:>4}  "
            f"{fill.order.shares:>4}  {fill.order.symbol:<6} "
            f"{fill.filled_price:>8.2f}"
        )
else:
    print("无交易记录")

日期                          方向    数量  标的          成交价
-------------------------------------------------------
2023-10-17 00:00:00        BUY   100  AAPL     175.10
2023-10-23 00:00:00       SELL   100  AAPL     171.00
2023-11-10 00:00:00        BUY   100  AAPL     184.48
2024-01-09 00:00:00       SELL   100  AAPL     183.24
2024-01-30 00:00:00        BUY   100  AAPL     186.11
2024-02-05 00:00:00       SELL   100  AAPL     185.75
2024-05-06 00:00:00        BUY   100  AAPL     180.07
2024-08-12 00:00:00       SELL   100  AAPL     216.11
2024-08-19 00:00:00        BUY   100  AAPL     224.42
2024-09-13 00:00:00       SELL   100  AAPL     221.05
2024-09-23 00:00:00        BUY   100  AAPL     224.99
2024-11-11 00:00:00       SELL   100  AAPL     223.01
2024-11-25 00:00:00        BUY   100  AAPL     231.60


---
## 9. 查看宽表

`result.mktdata` 保留了完整的宽表，可以直接观察 Indicator 和 Signal 的计算结果：

In [10]:
df = result.mktdata["AAPL"]
print(f"宽表列: {list(df.columns)}")
print(f"总行数: {len(df)}")
print()

# 显示信号触发点附近的数据
signal_days = df[df["sma_10_x_sma_50"] == True]
print(f"金叉触发次数: {len(signal_days)}")
if not signal_days.empty:
    print()
    print("金叉触发日的宽表数据：")
    print(signal_days[["close", "sma_10", "sma_50", "sma_10_x_sma_50"]])

宽表列: ['open', 'high', 'low', 'close', 'volume', 'sma_10', 'sma_50', 'sma_10_x_sma_50']
总行数: 501

金叉触发次数: 7

金叉触发日的宽表数据：
                 close      sma_10      sma_50  sma_10_x_sma_50
date                                                           
2023-10-17  175.097900  175.806609  175.718195             True
2023-11-10  184.483490  176.160033  174.434605             True
2024-01-30  186.106598  189.313316  188.962170             True
2024-05-06  180.071213  171.079034  170.817614             True
2024-08-19  224.415848  216.855614  216.609894             True
2024-09-23  224.992050  221.085707  220.804152             True
2024-11-25  231.604813  226.674747  226.636324             True


---
## 10. 分阶段执行（Partial Execution）

引擎支持 `run_through` 参数，在任意阶段终止执行。这对逐组件独立评估非常有用——先验证 Indicator 是否合理，再看 Signal 是否有预测力，最后才加入 Rule 和仓位管理。

In [ ]:
# 只执行到 Indicator 阶段
sim_broker_ind = SimBroker()
result_ind = engine.run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=sim_broker_ind,
    start="2023-01-01",
    end="2024-12-31",
    run_through="indicator",
)

df_ind = result_ind.mktdata["AAPL"]
print(f"Indicator 阶段 — 宽表列: {list(df_ind.columns)}")
print(f"交易次数: {len(result_ind.trades)}  (预期为 0)")
print()

# SMA 值预览
print(df_ind[["close", "sma_10", "sma_50"]].tail())

In [ ]:
# 只执行到 Signal 阶段
sim_broker_sig = SimBroker()
result_sig = engine.run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=sim_broker_sig,
    start="2023-01-01",
    end="2024-12-31",
    run_through="signal",
)

df_sig = result_sig.mktdata["AAPL"]
print(f"Signal 阶段 — 宽表列: {list(df_sig.columns)}")
print(f"交易次数: {len(result_sig.trades)}  (预期为 0)")
print()

# 信号统计
n_signals = df_sig["sma_10_x_sma_50"].sum()
print(f"金叉信号总次数: {n_signals}")

---
## 11. 三接口架构详解

Engine 通过三个 Protocol 接口与外部世界交互，这是策略与执行环境解耦的关键：

| Protocol | 职责 | 方法 |
|----------|------|------|
| `MarketDataProvider` | 提供行情数据 | `get_bars()`, `get_latest()` |
| `OrderRouter` | 接收并路由订单 | `submit_order()` |
| `FillReceiver` | 返回成交结果 | `get_fills()` |

SimBroker 同时实现了 OrderRouter 和 FillReceiver 两个 Protocol，因此可以作为 `broker` 参数传入：

In [13]:
from oxq.core import OrderRouter, FillReceiver
from oxq.trade import SimBroker

broker = SimBroker()
print(f"SimBroker 满足 OrderRouter: {isinstance(broker, OrderRouter)}")
print(f"SimBroker 满足 FillReceiver: {isinstance(broker, FillReceiver)}")

SimBroker 满足 OrderRouter: True
SimBroker 满足 FillReceiver: True


未来切换到实盘时，只需替换 Provider，策略代码不变：

```python
# 回测：历史数据 + 模拟撮合
engine.run(strategy, market=LocalMarketDataProvider(),
           broker=sim_broker, ...)

# 模拟盘（未来）：实时数据 + 模拟撮合
engine.run(strategy, market=RealtimeDataProvider(),
           broker=sim_broker, ...)

# 实盘（未来）：实时数据 + 真实券商
engine.run(strategy, market=RealtimeDataProvider(),
           broker=live_broker, ...)
```

---
## 12. 多标的策略

同一套策略定义，换一个 Universe 就变成多标的策略，引擎代码零修改：

In [ ]:
# 下载更多标的
for symbol in ["MSFT", "GOOGL"]:
    downloader.download(symbol, start="2023-01-01", end="2024-12-31")

# 只改 Universe，其余不变
multi_strategy = Strategy(
    name="sma_crossover_multi",
    hypothesis="短期均线上穿长期均线的标的在后续持有期内有正超额收益",
    universe=StaticUniverse(("AAPL", "MSFT", "GOOGL")),
    indicators={
        "sma_10": (SMA(), {"period": 10}),
        "sma_50": (SMA(), {"period": 50}),
    },
    signals={
        "sma_10_x_sma_50": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
    },
    entry_rules=[EntryRule(signal="sma_10_x_sma_50", shares=100)],
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

sim_broker_multi = SimBroker()
result_multi = engine.run(
    multi_strategy,
    market=LocalMarketDataProvider(),
    broker=sim_broker_multi,
    start="2023-01-01",
    end="2024-12-31",
)

print(f"总收益率:   {result_multi.total_return():.2%}")
print(f"Sharpe Ratio: {result_multi.sharpe_ratio():.2f}")
print(f"最大回撤:   {result_multi.max_drawdown():.2%}")
print(f"交易次数:   {len(result_multi.trades)}")
print()

# 按标的分组显示交易
from collections import Counter
trade_counts = Counter(f.order.symbol for f in result_multi.trades)
for symbol, count in sorted(trade_counts.items()):
    print(f"  {symbol}: {count} 笔交易")

---
## 13. 三种买入规则对比

前面的 `EntryRule` 每次固定买入 100 股，不管股价高低。资金利用率很低——10 万资金只买了 ~2 万的仓位。

open-xquant 提供三种 EntryRule，覆盖从保守到激进的仓位管理需求：

| 规则 | 参数 | 买入逻辑 | 示例（市价 200，资金 10 万） |
|------|------|----------|---------------------------|
| `EntryRule` | signal, shares | 固定股数 | 买 100 股 = 2 万 |
| `TargetValueEntryRule` | signal, target_value | 按目标市值 | 买 400 股 = 8 万 |
| `FullPositionEntryRule` | signal | 全仓买入 | 买 500 股 = 10 万 |

In [15]:
from oxq.rules import TargetValueEntryRule, FullPositionEntryRule

# --- TargetValueEntryRule: 按目标市值买入 ---
rule_tv = TargetValueEntryRule(signal="sma_10_x_sma_50", target_value=50_000)
row = pd.Series({"close": 200.0, "sma_10_x_sma_50": True})

# 无持仓: 50000 / 200 = 250 股
order = rule_tv.evaluate("AAPL", row, Portfolio(cash=100_000.0))
print(f"TargetValue — 无持仓:  买入 {order.shares} 股")

# 已有 100 股: 补买 250-100=150 股
order2 = rule_tv.evaluate("AAPL", row, Portfolio(
    cash=80_000.0,
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=180.0)},
))
print(f"TargetValue — 有100股: 补买 {order2.shares} 股")

print()

# --- FullPositionEntryRule: 全仓买入 ---
rule_fp = FullPositionEntryRule(signal="sma_10_x_sma_50")

# 10万全仓: 100000 / 200 = 500 股
order3 = rule_fp.evaluate("AAPL", row, Portfolio(cash=100_000.0))
print(f"FullPosition — 10万现金: 买入 {order3.shares} 股")

# 卖出后只剩 3 万: 30000 / 200 = 150 股
order4 = rule_fp.evaluate("AAPL", row, Portfolio(cash=30_000.0))
print(f"FullPosition — 3万现金:  买入 {order4.shares} 股")

TargetValue — 无持仓:  买入 250 股
TargetValue — 有100股: 补买 150 股

FullPosition — 10万现金: 买入 500 股
FullPosition — 3万现金:  买入 150 股


### 回测对比：三种买入规则

用同一套指标和信号，只替换 EntryRule，对比三种买入方式的效果：

In [ ]:
# 公共组件
common = dict(
    hypothesis="SMA10 金叉 SMA50 买入，死叉卖出",
    universe=StaticUniverse(("AAPL",)),
    indicators={
        "sma_10": (SMA(), {"period": 10}),
        "sma_50": (SMA(), {"period": 50}),
    },
    signals={
        "sma_10_x_sma_50": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
    },
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

# 三种买入规则
strategies = {
    "固定100股": Strategy(
        name="fixed", entry_rules=[EntryRule(signal="sma_10_x_sma_50", shares=100)], **common,
    ),
    "目标市值8万": Strategy(
        name="target", entry_rules=[TargetValueEntryRule(signal="sma_10_x_sma_50", target_value=80_000)], **common,
    ),
    "全仓买入": Strategy(
        name="full", entry_rules=[FullPositionEntryRule(signal="sma_10_x_sma_50")], **common,
    ),
}

# 分别回测
results = {}
for label, strat in strategies.items():
    broker = SimBroker()
    r = Engine().run(
        strat,
        market=LocalMarketDataProvider(),
        broker=broker,
        start="2023-01-01",
        end="2024-12-31",
        initial_cash=100_000.0,
    )
    results[label] = r

# 对比
header = f"{'':>14}" + "".join(f"{label:>14}" for label in results)
print(header)
print("-" * len(header))
for metric, fn in [
    ("总收益率", lambda r: f"{r.total_return():.2%}"),
    ("Sharpe", lambda r: f"{r.sharpe_ratio():.2f}"),
    ("最大回撤", lambda r: f"{r.max_drawdown():.2%}"),
    ("交易次数", lambda r: f"{len(r.trades)}"),
    ("期末总资产", lambda r: f"{r.equity_curve[-1][1]:,.0f}"),
]:
    vals = "".join(f"{fn(r):>14}" for r in results.values())
    print(f"{metric:>14}{vals}")

---
## 小结

本教程覆盖了策略执行管道的核心概念：

| 组件 | 职责 | 计算模式 |
|------|------|----------|
| `SMA` | 计算移动平均线 | 向量化，per symbol |
| `Crossover` | 检测上穿信号 | 向量化，cross-sectional |
| `EntryRule` | 信号触发时买入（固定股数） | 逐 bar，有状态 |
| `TargetValueEntryRule` | 信号触发时买入（按目标市值） | 逐 bar，有状态 |
| `FullPositionEntryRule` | 信号触发时买入（全部现金） | 逐 bar，有状态 |
| `ExitRule` | 快线跌破慢线时卖出 | 逐 bar，有状态 |
| `Strategy` | 声明式策略定义 | — |
| `Engine` | 执行四阶段管道（provider-agnostic） | — |
| `SimBroker` | 模拟撮合（Broker） | — |
| `RunResult` | 绩效指标 + 交易记录 + 宽表 | — |

**四阶段管道**：

```
Phase 0: Universe    → 确定标的池（StaticUniverse / FilterUniverse）
Phase 1: Indicator   → 向量化计算指标，追加为宽表新列
Phase 2: Signal      → 截面信号生成，追加为宽表新列
Phase 3: Rule        → 逐 bar 读取信号 + 持仓 → 生成订单 → Broker 提交并接收成交
```

**核心设计原则**：
- **策略定义与执行环境分离** — 三接口架构（MarketDataProvider、OrderRouter、FillReceiver）
- **一个 Engine，三种模式** — 回测 / 模拟盘 / 实盘只需更换 Provider
- Indicator/Signal 是纯函数，不修改 mktdata
- Rule 是状态机，读取持仓生成订单
- 宽表避免层间数据传递的复杂性
- `run_through` 支持逐组件独立评估